# Исключения в Python

В жизни каждого разработчика бывают моменты, когда код упал с ошибкой во время выполнения.

<img src="./_resources/exception_in_life.png" height="300">

\- Можно ли написать программу совсем без ошибок?

\- Можно, но такая программма, вероятно, будет тривиальна по своей сути.

<br>

В реальной жизни системы с каждым годом становятся всё сложнее, растёт количество и вариативность данных, систем, подсистем и надсистем тоже становится больше - отсюда проистекает **невозможность исчерпывающего тестирования и ненулевая вероятность что-то не учесть в коде**.



In [1]:
# Самый просто пример ошибки

numerator = int(input())
denominator = int(input())

print(numerator/denominator)

ValueError: invalid literal for int() with base 10: 'jdn'

Как вы уже знаете, в Python нельзя делить на 0. Стоит пользователю ввести 0, и программа выбрасывает ошибку ZeroDivisionError

Давайте подумаем, какие у нас в целом есть стратегии реагирования на некорректное поведение?


* **Вернуть какое-то значение по умолчанию**. Например, -1 в методе find() для str

In [2]:
my_str = "abcd"
my_str.find("q")

-1

* **Вернуть ошибку отдельным значением**. Например, в Golang мы ожидаем от функции не только результат, но и возможную ошибку.

    Такой подход известен под названием **«look before you leap»** (LBYL) - сразу появляется большое количество проверок if/else

* **Выбросить исключение**

In [3]:
if denominator == 0:
    raise ValueError("The denominator cannot be zero!")

ValueError: The denominator cannot be zero!

* **Использовать контейнеры и последовательности вызовов** - рекомендую ознакомиться со [статьёй](https://habr.com/ru/companies/oleg-bunin/articles/445234/) Никиты Соболева.

In [ ]:
from returns.result import Result, Success, Failure

def divide(first: float, second: float) -> Result[float, ZeroDivisionError]:
    try:
        return Success(first / second)
    except ZeroDivisionError as exc:
        return Failure(exc)

1 + divide(1, 0)

# Зачем нам вообще работа с исключениями?

- Обработка ошибок (обработка ожидаемых ошибок)
- Уведомления о событиях (ключ не найден в словаре)
- Обработка особых ситуаций (случайные и очень редкие исключения, которые мы не хотим отдельно отлавливать)
- Заключительные операции
- Необычное управление потоком выполнения (пробовали когда-нибудь goto?)

# Что такое исключение в Python?

В питоне больше прижился подход **«easier to ask forgiveness than permission»** (EAFP). Он подразумевает ожидание выполнения программы таким образом, как будто ошибки произойти не может. Но при этом, если ошибка произошла, для ее обработки существует отдельный блок кода, который будет заниматься ее обработкой. Обработка ошибки не перемешивается с основной логикой работы программы.


- Код более читаемый.
- Уменьшается количество проверок.
- Нет возможности для race condition.

# Что нам понадобится для работы с исключениями в Python?

- **try/except|except\*** - перехватывает исключения, возбужденные интерпретатором или вашим программным кодом, и выполняет восстановительные операции.

- **try/finally** - выполняет заключительные операции независимо от того, возникло исключение или нет.

- **raise/assert** - дает возможность возбудить исключение программно.

- **with/as** - дает возможность возбудить исключение программно, при выполнении определенного условия.

Общий способ записи:

```python
try:
    <код, который может вызвать исключение>
except <КлассИсключения_1>:
    <обработка ошибки этого типа>
except <КлассИсключения_2>:
    <обработка другого типа ошибки>
...
else:
    <выполняется, если исключений не было>
finally:
    <выполняется всегда -- и при ошибках, и без>
```

# Анатомия исключений

In [52]:
exception = Exception("Hello", "my", "friend")

print(exception.args)

('Hello', 'my', 'friend')


In [ ]:
print(exception.__cause__)
print(exception.__context__)
print(exception.__traceback__)

None
None


**\_\_cause\_\_** - причина нашего исключения (буквально экземпляр исключения-источника).

**\_\_context\_\_** - другие исключения, которые произошли до текущего.

**\_\_traceback\_\_** - стек вызовов

In [54]:
exception.add_note("My_note!")

raise exception

Exception: ('Hello', 'my', 'friend')

Что мы здесь видим?
1. Место, где произошло исключение
2. Тип исключения
3. Аргументы исключения
4. Дополнительные заметки

# Как выбрасываются исключения?

Обратимся к [документации](https://docs.python.org/3/reference/simple_stmts.html#the-raise-statement):

> If no expressions are present, raise re-raises the exception that is currently being handled, which is also known as the active exception.
> If there isn’t currently an active exception, a RuntimeError exception is raised indicating that this is an error.

>Otherwise, raise evaluates the first expression as the exception object. It must be either a subclass or an instance of BaseException. If it is a class, the exception instance will be obtained when needed by instantiating the class with no arguments.

In [16]:
raise

RuntimeError: No active exception to reraise

In [ ]:
raise ValueError

ValueError: 

Выбросить какой угодно объект не получится - `raise` работает только с потомками `BaseException`:

In [18]:
raise 5

TypeError: exceptions must derive from BaseException

Немногие вспомнят, но в Python 2 когда-то можно было выбрасывать ещё и строковые исключения:

<img src="./_resources/strexc.png" width=500>

Ещё один способ возбудить исключение - использовать `assert`:

In [20]:
assert "hello" == None

AssertionError: 

Кстати, существует способ запустить программу в режиме игнорирования таких проверок: "python -O example.py"

# Как ловить исключения через try-except?

In [21]:
try:
    print(1/0)
except Exception:  # До ZeroDivisionError уже не дойдём.
    print("Exception")
except ZeroDivisionError:
    print("ZeroDivisionError")

Exception


Порядок исключений имеет значение: **сначала пишем частные случаи, потом общие типы** (например, Exception)

А почему?

<img src="./_resources/hierachy.png" height=600>

Базовый класс здесь - `BaseException`. От него наследуется `Exception`, `GeneratorExit`, `SystemExit` и `KeyboardInterrupt`.

Вы, вероятно, замечали, что линтеры не рекомендуют использовать bare `except`.

Дело в том, что except в данном случае будет отлавливать все BaseException based исключения, в том числе SystemExit и KeyboardInterrupt.

Так делать плохо!

In [ ]:
# Пример плохого кода

try:
    # Какая-то логика
    pass
except:
    print("Поймали ошибку!")

Плохо ещё и потому, что такой глобальный обработчик может отловить даже те ошибки, для которых по-хорошему должно быть отдельное поведение. Это может привести к тому, что ваша программа не падает, но работает при этом всё равно некорректно. А вы об этом даже и не узнаете. Падать, так падать!

Поэтому ловим конкретные исключения и корректно их обрабатываем.

Кстати, в обработчике ещё можно обратиться к экземпляру пойманного исключения:

In [37]:
try:
    print(1/0)
except ZeroDivisionError as ex:  # Можем положить экземпляр исключения в переменную!
    print(f"Handled Error: {ex}")
except Exception:
    print("Error")

Handled Error: division by zero


### Перевыброс исключений

Иногда бывают ситуации, когда в случае определённого исключения нужно что-то сделать, но само исключение подавлять не нужно. Тогда исключение можно перевыбросить!

In [4]:
try:
    print(1/0)
except ZeroDivisionError as ex:
    print(f"Handled Error: {ex}")
    # Тут какая-то логика
    raise
except Exception:
    print("Error")

Handled Error: division by zero


ZeroDivisionError: division by zero

### Цепочка исключений

Если в блоке `except` тоже выбрасывается исключение (не всегда намеренно), исключения [связываются](https://docs.python.org/3/tutorial/errors.html#exception-chaining) в цепочку:

In [62]:
def test_chain():
    try:
        print(1/0)
    except ZeroDivisionError:
        raise RuntimeError("Big problem, BOSS")
    except Exception:
        print("ZeroDivisionError")

test_chain()

RuntimeError: Big problem, BOSS

#### Инструкция raise from

In [78]:
def process_data_simple(data):
    try:
        num = int(data)
        result = 1 / num
        return result
    except ValueError as ve:
        new_exc = TypeError("Data should be a valid number for processing")
        raise new_exc
    except ZeroDivisionError as zde:
        new_exc = ArithmeticError("Division by zero is not allowed in this context")
        raise new_exc


def process_data(data):
    try:
        num = int(data)
        result = 1 / num
        return result
    except ValueError as ve:
        new_exc = TypeError("Data should be a valid number for processing")
        raise new_exc from ve
    except ZeroDivisionError as zde:
        new_exc = ArithmeticError("Division by zero is not allowed in this context")
        raise new_exc from zde

In [72]:
import traceback

In [73]:
try:
    process_data_simple('abc')
except TypeError as te:
    print(f"Caught TypeError: {te}")
    print(f"Original exception: {te.__cause__}")
    print(f"Type of __context__: {type(te.__context__)}")
    print(f"Exception context: {te.__context__}")

Caught TypeError: Data should be a valid number for processing
Original exception: None
Type of __context__: <class 'ValueError'>
Exception context: invalid literal for int() with base 10: 'abc'


In [79]:
try:
    process_data('abc')
except TypeError as te:
    print(f"Caught TypeError: {te}")
    print(f"Type of __cause__: {type(te.__cause__)}")
    print(f"Original exception: {te.__cause__}")
    print(f"Exception context: {te.__context__}")
    traceback.print_exception(te) # Попробуем from ve заменить на from None

Caught TypeError: Data should be a valid number for processing
Type of __cause__: <class 'ValueError'>
Original exception: invalid literal for int() with base 10: 'abc'
Exception context: invalid literal for int() with base 10: 'abc'


Traceback (most recent call last):
  File "/var/folders/j4/6xsrj7l954v0777shqrvqnwm0000gn/T/ipykernel_91387/3841192051.py", line 16, in process_data
    num = int(data)
ValueError: invalid literal for int() with base 10: 'abc'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/var/folders/j4/6xsrj7l954v0777shqrvqnwm0000gn/T/ipykernel_91387/1543862588.py", line 2, in <module>
    process_data('abc')
    ~~~~~~~~~~~~^^^^^^^
  File "/var/folders/j4/6xsrj7l954v0777shqrvqnwm0000gn/T/ipykernel_91387/3841192051.py", line 21, in process_data
    raise new_exc from ve
TypeError: Data should be a valid number for processing


# Как работает try-finally

In [6]:
try:
    n = int(input("Please enter an integer: "))
    print(1 / n)
except (ZeroDivisionError, ValueError) as e:
    print(e)
finally:
    print("Finally work")

0.1
Finally work


Обычно блок `finally` используется для завершения операций — например, чтобы закрыть файл или освободить ресурсы.

В Java, например, есть "try with resources" - с такими же целями может использоваться.

<br>

Многие из вас писали в задачках ЕГЭ что-то такое:

```python
my_file = open("path_to_file/26.txt")
```

Но немногие потом писали в конце это:
```python
my_file.close()
```

Для открытия файла Python делает системный вызов к ОС, а та в свою очередь возвращает файловый дескриптор. Количество дескрипторов ограничено, нельзя бесконечно часто обращаться к памяти для чтения. Поэтому ресурсы нужно за собой закрывать.

Можно почитать [здесь](https://realpython.com/why-close-file-python/).


In [15]:
try:
    file = open("hello.txt", mode="w")
    file.write("Hello, World!")
finally:
    file.close()

In [14]:
def reverse_str(word: str) -> str:
    try:
        return word[::-1]
    finally:
        return "Хи-хи-хи-ха"

reverse_str("hello")

'Хи-хи-хи-ха'

В python 3.14 разработчики языка уже добавили SyntaxWarning для этого кейса!

Ошибка будет выглядеть так: **SyntaxWarning: 'return' in a 'finally' block**. Аналогично Python будет ругаться на `continue` и `break`.

# А что делать, если ошибок может быть сразу несколько за раз?

В Python 3.11 появились [группы](https://docs.python.org/3/tutorial/errors.html#raising-and-handling-multiple-unrelated-exceptions) исключений: **ExceptionGroup(msg, excs)** и **BaseExceptionGroup(msg, excs)**


- BaseExceptionGroup расширяет BaseException и может обертывать любое исключение.
- ExceptionGroup расширяет Exception и может обертывать только подклассы Exception.

In [27]:
def f():
    excs = [OSError('error 1'), SystemError('error 2')]
    raise ExceptionGroup('there were problems', excs)

f()

  + Exception Group Traceback (most recent call last):
  |   File "/Users/nemo/mipt_homeworks_2026/.venv/lib/python3.13/site-packages/IPython/core/interactiveshell.py", line 3747, in run_code
  |     exec(code_obj, self.user_global_ns, self.user_ns)
  |     ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  |   File "/var/folders/j4/6xsrj7l954v0777shqrvqnwm0000gn/T/ipykernel_91387/1521594820.py", line 5, in <module>
  |     f()
  |     ~^^
  |   File "/var/folders/j4/6xsrj7l954v0777shqrvqnwm0000gn/T/ipykernel_91387/1521594820.py", line 3, in f
  |     raise ExceptionGroup('there were problems', excs)
  | ExceptionGroup: there were problems (2 sub-exceptions)
  +-+---------------- 1 ----------------
    | OSError: error 1
    +---------------- 2 ----------------
    | SystemError: error 2
    +------------------------------------


In [28]:
try:
    f()
except Exception as e:
    print(f'caught {type(e)}: {e}')

caught <class 'ExceptionGroup'>: there were problems (2 sub-exceptions)


Группы исключений можно распаковывать и обрабатывать вложенные в них исключения по их типам.

Для этого используется слово `except*`. Если нужный тип есть, из группы убираются все (в том числе вложенные) исключения этого типа.

In [ ]:
def f():
    raise ExceptionGroup(
        "group1",
        [
            OSError(1),
            SystemError(2),
            ExceptionGroup(
                "group2",
                [
                    OSError(3),
                    # RecursionError(4)
                ]
            )
        ]
    )


try:
    f()
except* OSError as e:
    print("There were OSErrors")
except* SystemError as e:
    print("There were SystemErrors")

There were OSErrors
There were SystemErrors


Комбинировать `except` и `except*` нельзя!

In [32]:
try:
    f()
except* OSError as e:
    print("There were OSErrors")
except* SystemError as e:
    print("There were SystemErrors")
except Exception as e:
    print(e)

SyntaxError: cannot have both 'except' and 'except*' on the same 'try' (1010498803.py, line 7)

In [7]:
from dataclasses import dataclass


@dataclass
class UserInput:
    email: str
    age: int
    username: str


def validate_user(data: UserInput) -> None:
    errors: list[Exception] = []

    if "@" not in data.email:
        errors.append(ValueError(f"Invalid email: {data.email!r}"))
    if not 13 <= data.age <= 120:
        errors.append(ValueError(f"Age {data.age} out of range [13, 120]"))
    if len(data.username) < 3:
        errors.append(ValueError(f"Username too short: {data.username!r}"))
    if not data.username.isalnum():
        errors.append(ValueError(f"Username must be alphanumeric: {data.username!r}"))

    if errors:
        raise ExceptionGroup("Validation failed", errors)


try:
    validate_user(UserInput(email="bad", age=5, username="a!"))
except* ValueError as eg:
    for err in eg.exceptions:
        print(f"  - {err}")

  - Invalid email: 'bad'
  - Age 5 out of range [13, 120]
  - Username too short: 'a!'
  - Username must be alphanumeric: 'a!'


In [ ]:
example = Exception("a", "b")

str(example)

"('a', 'b')"

# Какие есть минусы у работы с исключениями?

In [1]:
def divide(first: float, second: float) -> float:
    return first / second

ZeroDivisionError нужно как-то обработать. Давайте отловим его и вернём значение по умолчанию:

In [ ]:
def divide(first: float, second: float) -> float:
    try:
        return first / second
    except ZeroDivisionError:
        return 0.0

Но почему мы возвращаем 0? Почему не 1 или None?  Конечно, в большинстве случаев, получить None почти так же плохо (если даже не хуже), как исключение, но все же нужно опираться на бизнес-логику и варианты использования функции.

Что именно мы делим? Произвольные числа, какие-то конкретные единицы или деньги?

* Если возвращать None, то код с вызовами этого метода будет обрастать бесконечными проверками if result is not None:

А если не отлавливать исключение внутри функции? Пусть их отлавливают при вызове функции!

Но тогда есть несколько моментов:
* А точно ли эту ошибку отловят? А на каком уровне?
* А не упрёмся в пустой try except непонятно где?

## А что за Warnings были на схеме классов?

Обратите внимание, что на схеме с типами исключений также есть класс **Warning**. Этот класс добавил ещё Гвидо ван Россум в 2000 году)

В отличии от ислючений предупреждения не прерывают выполнение кода, однако позволяют предупредить разработчика о некоторых узких местах: например, о скором выпиливании фичи или о том. Часто warning ещё можно увидеть, если вы делаете http запрос без SSL сертификатов.

In [ ]:
import warnings
import math

def calculate():
    try:
        result = 1 / 0
    except ZeroDivisionError:
        warnings.warn("Division by zero occurred", RuntimeWarning)
        result = math.nan
    return result

result = calculate()

/var/folders/j4/6xsrj7l954v0777shqrvqnwm0000gn/T/ipykernel_91387/831935586.py:8: RuntimeWarning: Division by zero occurred
  warnings.warn("Division by zero occurred", RuntimeWarning)


Разные типы Warning нужны для последующей фильтрации предупреждений:

In [ ]:
warnings.simplefilter('ignore') # Игнорируем предупреждения

print(calculate())

nan


In [ ]:
warnings.simplefilter('error') # Пусть все предупреждения будут выбрасывать ошибку!

print(calculate())

RuntimeWarning: Division by zero occurred

In [ ]:
# Есть и более продвинутая фильтрация

warnings.filterwarnings("ignore", category=DeprecationWarning, module="my_module", message="Some Message!")

In [ ]:
warnings.resetwarnings()  # Сбросили все фильтры

Вот все режимы для предупреждений:
- “default” — display a warning the first time it is encountered
- “error” — turn the warning into an exception
- “ignore” — ignore the warning
- “always” — always show the warning, even if it was displayed before
- “module” — show the warning once per module
- “once” — show the warning only once, throughout the program

Рекомендую почитать вот эту [статью](https://coderivers.org/blog/python-warnings/) и [документацию](https://docs.python.org/3/library/warnings.html).

А вот так обычно поступают разработчики библиотек, когда старую реализацию хотят удалить в будущих версиях:

In [ ]:
import warnings

def old_function():
    warnings.warn("old_function is deprecated. Use new_function instead", DeprecationWarning)
    return "Old function result"

def new_function():
    return "New function result"

# Обработка исключений через менеджеры контекста (контекстные менеджеры)

Выше мы посмотрели на конструкцию try - finally. Давайте представим, что наша программма читает файл на компьютере пользователя:

In [ ]:
file = open("file.txt", "r")
try:
    # Действия с файлом
    content = file.read()
    print(content)
finally:
    file.close()

Чтобы упростить эту конструкцию можно прибегнуть к использованию менеджеров контекста:

In [ ]:
with open("file.txt", "r") as file:
    content = file.read()
    print(content)

Независимо от того, как выполнится код внутри блока `with`, для открытого файла будет вызван метод `close()`.

Подобных примеров множество:

In [ ]:
import sqlite3

with sqlite3.connect('example.db') as conn:
    cursor = conn.cursor()

In [ ]:
import threading

with threading.Lock() as lock:
    ...

Чтобы создать свой контекстный менеджер в Python, вам необходимо определить класс, который содержит методы __enter__() и __exit__().

Метод __enter__() выполняется перед выполнением блока кода внутри оператора with. Он может выполнять какие-либо подготовительные действия или возвращать значение, которое будет связано с переменной после ключевого слова as.

Метод __exit__() вызывается после завершения выполнения блока кода with. Он используется для выполнения завершающих действий, таких как освобождение ресурсов, обработка исключений или выполнение финализирующих операций.

Вот пример простого контекстного менеджера, который записывает время выполнения блока кода:

In [ ]:
import time

class Timer:
    def __enter__(self):
        self.start_time = time.time()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        elapsed_time = time.time() - self.start_time
        print(f"Elapsed time: {elapsed_time} seconds")

И теперь можем использовать вот такую красоту:

In [ ]:
with Timer() as timer:
    time.sleep(2)

Elapsed time: 2.0050289630889893 seconds


In [ ]:
# А без синтаксического сахара это выглядит так:

timer_manager = Timer()
timer_manager.__enter__()
try:
    # Делаем работу
    pass
finally:
    timer_manager.__exit__(None, None, None)

Аргументы `exc_type`, `exc_value`, `traceback` в __exit__() позволяют узнать, было ли исключение в блоке with.

Можно на основе этого принять решение: подавить исключение (вернув True) или дать ему распространиться дальше.

In [8]:
with Timer() as timer:
    raise OSError("File Not Found!")

Elapsed time: 4.0531158447265625e-06 seconds


Кстати, контекстных менеджеров может быть сразу несколько:

In [ ]:
# Можно, конечно, так:
with open("my_file.txt", "w"):
    with Timer() as timer:
        pass

# Но лучше так:
with open("my_file.txt", "w"), Timer() as timer:
    pass

Существует и альтернативный способ: использовать встроенный пакет contextlib:

In [ ]:
from contextlib import contextmanager

@contextmanager
def timer():
    start_time = time.time()
    try:
        yield # Про yield поговорим через пару занятий.
    finally:
        elapsed_time = time.time() - start_time
        print(f"Elapsed time: {elapsed_time} seconds")


with timer() as timer:
    time.sleep(2)

Elapsed time: 2.001605749130249 seconds


## Фишки contextlib

`contextlib.suppress (*exceptions)`: менеджер контекста, подавляющий перечисленные исключения.

In [ ]:
from contextlib import suppress

with suppress(FileNotFoundError):
    with open('missing_file.txt') as f:
        data = f.read()
    # Исключение FileNotFoundError будет подавлено

`contextlib.redirect_stdout()` и `contextlib.redirect_stderr()`: перенаправляют стандартный вывод или ошибок на другой поток или файл.

In [ ]:
from contextlib import redirect_stdout
import io

buffer = io.StringIO()
with redirect_stdout(buffer):
    print("Hello, World!")

# Все, что печаталось в print, теперь оказалось в buffer
print(buffer.getvalue())  # "Hello, World!\n"

`contextlib.ExitStack()`: позволяет динамически управлять множеством контекстов, удобно если количество менеджеров контекста заранее неизвестно.

In [ ]:
from contextlib import ExitStack

files = ['file1.txt', 'file2.txt']
with ExitStack() as stack:
    # открываем файлы динамически
    file_objects = [stack.enter_context(open(fname, 'r')) for fname in files]
    # если возникнет исключение, все уже открытые файлы будут закрыты
    for f in file_objects:
        print(f.read())

# Пользовательские исключения

Пользовательские исключения [нужно](https://docs.python.org/3/library/exceptions.html#BaseException) делать на основе Exception.

Так как был запрос на архитектуру, попробуем чуть копнуть в этом направлении (опять не банда четырёх...)

Исходники кода доступны [здесь](https://github.com/ArjanCodes/examples/tree/main/2026/ports).

Очень рекомендую оригинальный [ролик](https://www.youtube.com/watch?v=FXwBWS4qDAA) и канал ArjanCodes в целом.

In [ ]:
# api.py

from typing import Any

from db import get_db
from fastapi import APIRouter, Depends, HTTPException
from pydantic import BaseModel, Field
from sqlalchemy import text
from sqlalchemy.engine import Connection

router = APIRouter()


class PlaceOrderIn(BaseModel):
    sku: str
    qty: int = Field(..., gt=0)


class PlaceOrderOut(BaseModel):
    sku: str
    qty: int
    remaining_stock: int


def place_order(db: Connection, sku: str, qty: int) -> dict[str, Any]:
    if qty <= 0:
        raise HTTPException(status_code=400, detail="qty must be > 0")

    row = db.execute(
        text("SELECT stock FROM inventory WHERE sku=:sku"),
        {"sku": sku},
    ).fetchone()

    if row is None:
        raise HTTPException(status_code=404, detail="unknown sku")

    available = int(row.stock)
    if available < qty:
        raise HTTPException(
            status_code=409,
            detail=f"out of stock: requested {qty}, available {available}",
        )

    db.execute(
        text("UPDATE inventory SET stock = stock - :qty WHERE sku=:sku"),
        {"sku": sku, "qty": qty},
    )
    remaining = db.execute(
        text("SELECT stock FROM inventory WHERE sku=:sku"),
        {"sku": sku},
    ).scalar_one()

    # API response shaping in the domain-y function
    return {"status": "ok", "sku": sku, "qty": qty, "remaining_stock": int(remaining)}


@router.post("/orders", response_model=PlaceOrderOut)
def place_order_endpoint(
    payload: PlaceOrderIn,
    connection: Connection = Depends(get_db),
) -> PlaceOrderOut:
    result = place_order(connection, payload.sku, payload.qty)
    return PlaceOrderOut(**result)

Давайте представим, что мы разрабатываем Backend для какого-то магазина. Пользователю нужно предоставить возможность создавать заказы. От него мы ожидаем идентификатор товара (sku - Stock Keeping Unit) и количество (qty - Quantity).

Наша задача - проверить, что такой товар существует и запрашиваемое количество единиц есть на складе.

Как можем видеть по коду - тут всё намешано в одну кучу: запросы в базу, бизнес-логика, код для работы с HTTP...

Верный признак, что что-то не так - это импорты. Наше API - это просто интерфейс для внешних HTTP-запросов, этот код вообще не должен быть в курсе, как у нас хранятся данные: хоть в файле, хоть в таблице, хоть случайно генерируются.

Представьте, что вам понадобилось изменить название таблицы в базе? Почему вдруг это должно влиять на API?

## Domain + Ports + Adapters

In [ ]:
# domain/errors.py

class DomainError(Exception):
    pass

class InvalidQuantity(DomainError):
    pass

class UnknownSku(DomainError):
    def __init__(self, sku: str) -> None:
        super().__init__(f"Unknown sku: {sku}")
        self.squ = sku

class OutOfStock(DomainError):
    def __init__(self, sku: str, requested: int, available: int) -> None:
        super().__init__(
            f"out of stock: {sku}, requested {requested}, available {available}"
        )
        self.sku = sku
        self.requested = requested
        self.available = available


In [ ]:
# domain/models.py

from dataclasses import dataclass


@dataclass(frozen=True)
class OrderRequest:
    sku: str
    qty: int


@dataclass(frozen=True)
class OrderPlaced:
    sku: str
    qty: int
    remaining_stock: int

In [ ]:
# domain/ports.py

from typing import Protocol


class InventoryPort(Protocol):
    def exists_sku(self, sku: str) -> bool: ...
    def get_stock(self, sku: str) -> int: ...
    def reserve(self, sku: str, qty: int) -> int: ...

In [ ]:
# adapters/inventory.py

from dataclasses import dataclass

from domain.ports import InventoryPort
from sqlalchemy import text
from sqlalchemy.engine import Connection


@dataclass
class SqlAlchemyInventoryAdapter(InventoryPort):
    conn: Connection

    def exists_sku(self, sku: str) -> bool:
        row = self.conn.execute(
            text("SELECT 1 FROM inventory WHERE sku = :sku"),
            {"sku": sku},
        ).fetchone()

        return row is not None

    def get_stock(self, sku: str) -> int:
        row = self.conn.execute(
            text("SELECT stock FROM inventory WHERE sku = :sku"),
            {"sku": sku},
        ).fetchone()

        return int(row.stock)

    def reserve(self, sku: str, qty: int) -> int:
        self.conn.execute(
            text("UPDATE inventory SET stock = stock - :qty WHERE sku = :sku"),
            {"sku": sku, "qty": qty},
        )
        remaining = self.conn.execute(
            text("SELECT stock FROM inventory WHERE sku = :sku"),
            {"sku": sku},
        ).scalar_one()

        self.conn.commit()

        return int(remaining)

In [ ]:
# domain/use_cases.py

from .errors import InvalidQuantity, OutOfStock, UnknownSku
from .models import OrderPlaced, OrderRequest
from .ports import InventoryPort


def place_order(req: OrderRequest, inventory: InventoryPort) -> OrderPlaced:
    if req.qty <= 0:
        raise InvalidQuantity()

    if not inventory.exists_sku(req.sku):
        raise UnknownSku(req.sku)

    available = inventory.get_stock(req.sku)
    if available < req.qty:
        raise OutOfStock(req.sku, req.qty, available)

    remaining = inventory.reserve(req.sku, req.qty)
    return OrderPlaced(sku=req.sku, qty=req.qty, remaining_stock=remaining)

In [ ]:
# Refactored api.py

from adapters.sqlalchemy_inventory import SqlAlchemyInventoryAdapter
from db import get_db
from domain.errors import InvalidQuantity, OutOfStock, UnknownSku
from domain.models import OrderRequest
from domain.use_cases import place_order
from fastapi import APIRouter, Depends, HTTPException
from pydantic import BaseModel, Field
from sqlalchemy.engine import Connection

router = APIRouter()


class PlaceOrderIn(BaseModel):
    sku: str
    qty: int = Field(..., gt=0)


class PlaceOrderOut(BaseModel):
    sku: str
    qty: int
    remaining_stock: int


@router.post("/orders", response_model=PlaceOrderOut)
def place_order_endpoint(
    payload: PlaceOrderIn,
    connection: Connection = Depends(get_db),
) -> PlaceOrderOut:
    try:
        result = place_order(
            OrderRequest(**payload.model_dump()),
            SqlAlchemyInventoryAdapter(conn=connection),
        )
    except InvalidQuantity as e:
        raise HTTPException(status_code=400, detail=str(e)) from e
    except UnknownSku as e:
        raise HTTPException(status_code=404, detail=str(e)) from e
    except OutOfStock as e:
        raise HTTPException(status_code=409, detail=str(e)) from e

    return PlaceOrderOut(
        sku=result.sku,
        qty=result.qty,
        remaining_stock=result.remaining_stock,
    )